# Evaluación de un modelo de riesgo mediante diferentes umbrales

## Contexto

En este ejercicio se simula un escenario bancario en el que un modelo calcula la probabilidad de que un cliente represente un riesgo. El objetivo es evaluar cómo cambia la clasificación de los clientes al modificar el **umbral de decisión** utilizado por el modelo.

Primero se generan 100 valores reales, donde `0` representa un cliente que no representa riesgo y `1` representa un cliente que sí representa riesgo. Posteriormente, se generan probabilidades simuladas para cada cliente, las cuales representan la estimación realizada por el modelo.

La función `evaluar_umbral()` permite probar diferentes valores de umbral. Si la probabilidad estimada por el modelo es mayor que el umbral establecido, el cliente se clasifica como `1`; de lo contrario, se clasifica como `0`.

Para evaluar las predicciones se utiliza una **matriz de confusión**, que permite identificar:

- **Verdaderos Negativos (VN):** clientes correctamente clasificados como no riesgosos.
- **Falsos Positivos (FP):** clientes clasificados como riesgosos cuando realmente no lo son.
- **Falsos Negativos (FN):** clientes riesgosos que el modelo no detectó.
- **Verdaderos Positivos (VP):** clientes correctamente identificados como riesgosos.

Se probarán tres umbrales diferentes: **0.2, 0.5 y 0.8**, con el propósito de observar cómo cambia la cantidad de clientes rechazados y el riesgo de pérdida asociado a los falsos negativos.

Finalmente, se compararán los resultados de los diferentes umbrales para determinar cuál ofrece el menor riesgo de pérdida según los resultados obtenidos en la simulación.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix#para la matriz de confusion 

#set de ddatos de probabilidad de que un cleinte no pague a un banco 
np.random.seed(42)
#datos reales, 0 = pagar 80% pagaran, 20% no pagaran
y_real=np.random.choice([0,1],size=100,p=[0.8,0.2])
#                               genera 100 proba simuladas genera numeros entre 0 y 1, el 5 es el parametrso alfa y el 2 parametro beta, cuando el alfa es mayor a l parametros beta genera valores cercanos que estan a 1 
probabilidades=np.where(y_real==1,np.random.beta(5,2,100),np.random.beta(2,5,100))

df_banco=pd.DataFrame({
    
    'Valores_reales':y_real,
    'Probabilidad_modelo':probabilidades,
    
})

#funcion para evaluar el movimiento del umbral
#saber si la persona tiene cancer clasificacion 1, no tiene cancer 0
def evaluar_umbral(df,umbral):
    
    prediccion=(df['Probabilidad_modelo']>umbral).astype(int)
    cm=confusion_matrix(df['Valores_reales'],prediccion)
    #convierte la matriz en una sola linea de datos, el orden es 
    vn,fp,fn,vp=cm.ravel()#a cada variable 
    print(f"\nEvaluando Umbral de Desicion= {umbral}")
    print()
    print("--------------------------------------------------------------------------")
    print(f"Verdaderos Negativos (VN): {vn:2d} | Falsos Positivos (FP): {fp:2d}")
    print(f"Falsos Negativos (FN): {fn:2d}     | Verdaderos Positivos (VP): {vp:2d}")
    print("--------------------------------------------------------------------------")
    print()
    #cuantos clientes van a ser rechazados
    print(f"Clientes rechazados (Prediccion=1): {fp+vp}")
    print(f"Riesgo de Perdida (Fraudes No detectados (FN)): {fn}")
    
#prueba de cambio de umbrales
evaluar_umbral(df_banco,0.5)#umbral estandar 
evaluar_umbral(df_banco,0.2)# no arriesgar tando dinero
evaluar_umbral(df_banco,0.8)#banco agresivo, prestar a todo mundo
    


Evaluando Umbral de Desicion= 0.5

--------------------------------------------------------------------------
Verdaderos Negativos (VN): 74 | Falsos Positivos (FP):  8
Falsos Negativos (FN):  2     | Verdaderos Positivos (VP): 16
--------------------------------------------------------------------------

Clientes rechazados (Prediccion=1): 24
Riesgo de Perdida (Fraudes No detectados (FN)): 2

Evaluando Umbral de Desicion= 0.2

--------------------------------------------------------------------------
Verdaderos Negativos (VN): 35 | Falsos Positivos (FP): 47
Falsos Negativos (FN):  0     | Verdaderos Positivos (VP): 18
--------------------------------------------------------------------------

Clientes rechazados (Prediccion=1): 65
Riesgo de Perdida (Fraudes No detectados (FN)): 0

Evaluando Umbral de Desicion= 0.8

--------------------------------------------------------------------------
Verdaderos Negativos (VN): 82 | Falsos Positivos (FP):  0
Falsos Negativos (FN): 10     | Verdade

# Análisis de los resultados según el umbral de decisión

El modelo fue evaluado utilizando tres umbrales de decisión diferentes: **0.2, 0.5 y 0.8**. El objetivo es observar cómo cambia la clasificación de los clientes y, especialmente, la cantidad de **falsos negativos (FN)**, que representan los clientes riesgosos que el modelo no logró detectar.

## Comparación de los umbrales

| Umbral | VN | FP | FN | VP | Clientes rechazados | Riesgo de pérdida (FN) |
|:------:|---:|---:|---:|---:|--------------------:|-----------------------:|
| **0.2** | 35 | 47 | **0** | 18 | 65 | **0** |
| **0.5** | 74 | 8 | **2** | 16 | 24 | **2** |
| **0.8** | 82 | 0 | **10** | 8 | 8 | **10** |

### Umbral de 0.2

Con un umbral de **0.2**, el modelo clasifica como riesgosos a los clientes cuya probabilidad estimada sea mayor al 20%.

En este caso se obtuvieron **35 verdaderos negativos, 47 falsos positivos, 0 falsos negativos y 18 verdaderos positivos**.

El aspecto más importante es que se obtuvieron **0 falsos negativos**, por lo que ningún cliente que realmente representaba un riesgo quedó sin detectar. Sin embargo, el modelo fue más estricto y clasificó a **65 clientes como riesgosos**, de los cuales **47 fueron falsos positivos**.

### Umbral de 0.5

Con el umbral estándar de **0.5**, se obtuvieron **74 verdaderos negativos, 8 falsos positivos, 2 falsos negativos y 16 verdaderos positivos**.

En este escenario solamente se clasificaron **24 clientes como riesgosos**. El modelo redujo considerablemente los falsos positivos en comparación con el umbral de 0.2, pero aumentó los falsos negativos de **0 a 2**.

Esto representa un equilibrio entre detectar clientes riesgosos y evitar rechazar clientes que realmente no representan un riesgo.

### Umbral de 0.8

Con un umbral de **0.8**, el modelo solamente clasifica como riesgosos a los clientes con una probabilidad superior al 80%.

Se obtuvieron **82 verdaderos negativos, 0 falsos positivos, 10 falsos negativos y 8 verdaderos positivos**.

Aunque este umbral no genera falsos positivos, aumenta considerablemente los falsos negativos. Esto significa que **10 clientes que realmente representaban un riesgo no fueron detectados**.

## Comparación general

Los resultados muestran que modificar el umbral cambia el comportamiento del modelo:

- Al utilizar un umbral **bajo (0.2)**, se detectan más clientes riesgosos y se reducen los falsos negativos, pero aumenta considerablemente la cantidad de falsos positivos.
- Con un umbral **intermedio (0.5)**, se obtiene un equilibrio entre falsos positivos y falsos negativos.
- Con un umbral **alto (0.8)**, disminuyen los falsos positivos, pero aumenta considerablemente el número de falsos negativos.

## Conclusión

Para este ejercicio, se selecciona el **umbral de 0.2** debido a que presenta **0 falsos negativos**, lo que significa que no se dejaron clientes riesgosos sin detectar en la simulación.

Esta elección es especialmente importante en un escenario donde el **riesgo de pérdida por no detectar a un cliente riesgoso es prioritario**. Sin embargo, esta decisión implica aceptar un mayor número de falsos positivos: **47 clientes fueron clasificados como riesgosos cuando realmente no lo eran**.

Por lo tanto, el umbral de **0.2 minimiza el riesgo de clientes riesgosos no detectados**, mientras que el umbral de **0.8 minimiza los falsos positivos**. La elección final del umbral dependerá de cuál de estos dos tipos de error represente un mayor costo para la institución.